In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/behavior-aware-bms"

In [5]:
battery = pd.read_csv(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv"
)

print(battery.shape)

battery.head()

(34, 8)


,battery_id,avg_stress,avg_temp,fast_charge_duration,deep_discharge_duration,high_temp_duration,aggressive_discharge_count,avg_soc
0,B0005,13.670454,26.369701,1.250087e+09,1.277054e+07,3581775.549,45284,NaN
1,B0006,11.407560,26.429154,1.027942e+09,1.931800e+07,4680506.022,44512,NaN
2,B0007,0.065179,26.119363,6.213354e+05,1.428340e+07,4420739.468,195,NaN
3,B0018,14.449429,25.913199,6.962533e+08,1.054231e+07,0.000,32084,NaN
4,B0025,16.337941,28.482356,2.693701e+08,1.155482e+07,2827773.483,4349,NaN


In [6]:
print(battery.columns.tolist())

['battery_id', 'avg_stress', 'avg_temp', 'fast_charge_duration', 'deep_discharge_duration', 'high_temp_duration', 'aggressive_discharge_count', 'avg_soc']


In [7]:
battery["aging_budget"] = 100

In [8]:
battery["aging_budget"] -= np.where(
    battery["avg_stress"]>70,
    30,
    np.where(
        battery["avg_stress"]>50,
        20,
        10
    )
)

In [9]:
battery["aging_budget"] -= np.where(
    battery["avg_temp"]>40,
    25,
    np.where(
        battery["avg_temp"]>30,
        15,
        5
    )
)

In [10]:
battery["aging_budget"] -= np.where(
    battery["deep_discharge_duration"]>100,
    20,
    np.where(
        battery["deep_discharge_duration"]>20,
        10,
        5
    )
)

In [11]:
battery["aging_budget"] -= np.where(
    battery["fast_charge_duration"]>100,
    15,
    np.where(
        battery["fast_charge_duration"]>20,
        8,
        2
    )
)

In [12]:
battery["aging_budget"] -= np.where(
    battery["aggressive_discharge_count"]>500,
    15,
    np.where(
        battery["aggressive_discharge_count"]>100,
        8,
        2
    )
)

In [13]:
battery["aging_budget"] -= np.where(
    battery["avg_soc"]>80,
    10,
    np.where(
        battery["avg_soc"]<20,
        10,
        0
    )
)

In [14]:
battery["aging_budget"] = (
    battery["aging_budget"]
    .clip(0,100)
)

In [15]:
battery["stress_norm"] = (
    battery["avg_stress"]/100
)

In [16]:
battery["temp_norm"] = (
    battery["avg_temp"]/50
).clip(0,1)

In [17]:
battery["dd_norm"] = (
    battery["deep_discharge_duration"]/
    battery["deep_discharge_duration"].max()
)

In [18]:
battery["fc_norm"] = (
    battery["fast_charge_duration"]/
    battery["fast_charge_duration"].max()
)

In [19]:
battery["soc_norm"] = np.where(
    (battery["avg_soc"]<20)|
    (battery["avg_soc"]>80),
    1,
    0
)

In [20]:
battery["health_index"] = 100*(
      0.35*battery["stress_norm"]
    + 0.25*battery["temp_norm"]
    + 0.15*battery["dd_norm"]
    + 0.15*battery["fc_norm"]
    + 0.10*battery["soc_norm"]
)

In [21]:
def battery_state(x):

    if x>=80:
        return "CRITICAL"

    elif x>=60:
        return "DEGRADED"

    elif x>=40:
        return "WARNING"

    else:
        return "HEALTHY"


battery["battery_state"] = (
    battery["health_index"]
    .apply(battery_state)
)

In [22]:
battery[
    [
        "battery_id",
        "aging_budget",
        "health_index",
        "battery_state"
    ]
].head(20)

,battery_id,aging_budget,health_index,battery_state
0,B0005,35,33.237747,HEALTHY
1,B0006,35,29.947436,HEALTHY
2,B0007,42,13.389964,HEALTHY
3,B0018,35,26.589793,HEALTHY
4,B0025,35,23.434376,HEALTHY
5,B0026,35,23.066760,HEALTHY
6,B0027,48,15.079076,HEALTHY
7,B0028,35,15.329475,HEALTHY
8,B0029,15,40.248597,WARNING
9,B0030,15,39.842085,HEALTHY


In [23]:
health_dist = (
    battery["battery_state"]
    .value_counts()
)

print(health_dist)

battery_state
HEALTHY    33
WARNING     1
Name: count, dtype: int64


In [24]:
battery["health_index"] = 100 - battery["aging_budget"]

In [25]:
def battery_state(x):

    if x >= 80:
        return "CRITICAL"

    elif x >= 60:
        return "DEGRADED"

    elif x >= 30:
        return "WARNING"

    else:
        return "HEALTHY"

In [26]:
battery["health_index"] = (
    100 - battery["aging_budget"]
)

In [27]:
def battery_state(x):

    if x >= 80:
        return "CRITICAL"

    elif x >= 60:
        return "DEGRADED"

    elif x >= 30:
        return "WARNING"

    else:
        return "HEALTHY"

battery["battery_state"] = (
    battery["health_index"]
    .apply(battery_state)
)

In [28]:
health_dist = (
    battery["battery_state"]
    .value_counts()
)

print(health_dist)

battery_state
DEGRADED    16
WARNING     13
CRITICAL     5
Name: count, dtype: int64


In [29]:
battery["remaining_health"] = (
    battery["aging_budget"]
)

battery["consumed_life"] = (
    100 - battery["remaining_health"]
)

In [30]:
battery[
    [
        "battery_id",
        "remaining_health",
        "consumed_life",
        "health_index",
        "battery_state"
    ]
].head(20)

,battery_id,remaining_health,consumed_life,health_index,battery_state
0,B0005,35,65,65,DEGRADED
1,B0006,35,65,65,DEGRADED
2,B0007,42,58,58,WARNING
3,B0018,35,65,65,DEGRADED
4,B0025,35,65,65,DEGRADED
5,B0026,35,65,65,DEGRADED
6,B0027,48,52,52,WARNING
7,B0028,35,65,65,DEGRADED
8,B0029,15,85,85,CRITICAL
9,B0030,15,85,85,CRITICAL
